# Applied Machine Learning. Homework 4


**Group members:** *Sergi Cases and Martí Pascual* 



## Task 1 - Weather classification

Here we use the perceptron and the LinearSVC from SKlearn


In [1]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Perceptron as SklearnPerceptron
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline

X1 = [{'city': 'Gothenburg', 'month': 'July'},
      {'city': 'Gothenburg', 'month': 'December'},
      {'city': 'Paris',      'month': 'July'},
      {'city': 'Paris',      'month': 'December'}]
Y1 = ['rain', 'rain', 'sun', 'rain']

X2 = [{'city': 'Sydney', 'month': 'July'},
      {'city': 'Sydney', 'month': 'December'},
      {'city': 'Paris',  'month': 'July'},
      {'city': 'Paris',  'month': 'December'}]
Y2 = ['rain', 'sun', 'sun', 'rain']

clf1 = make_pipeline(DictVectorizer(), SklearnPerceptron(max_iter=10))
clf1.fit(X1, Y1)
print('Dataset 1 (Gothenburg / Paris) - train accuracy:',
      accuracy_score(Y1, clf1.predict(X1)))

clf2 = make_pipeline(DictVectorizer(), SklearnPerceptron(max_iter=10))
clf2.fit(X2, Y2)
print('Dataset 2 (Sydney / Paris)     - train accuracy:',
      accuracy_score(Y2, clf2.predict(X2)))

# Switching to LinearSVC does not help dataset 2 either:
clf2_svc = make_pipeline(DictVectorizer(), LinearSVC())
clf2_svc.fit(X2, Y2)
print('Dataset 2 with LinearSVC       - train accuracy:',
      accuracy_score(Y2, clf2_svc.predict(X2)))


Dataset 1 (Gothenburg / Paris) - train accuracy: 1.0
Dataset 2 (Sydney / Paris)     - train accuracy: 0.5
Dataset 2 with LinearSVC       - train accuracy: 0.5


### Discussion

The first step is to do one-hot encoding via DictVectorizer.

**Dataset 1 is linearly separable.**
Only one example (Paris in July) has the label sun; the other three are rain. It exists an hyperplane that is able to separate the classes. So when we train the perceptron can update the weights and find it.

All sun examples score above zero and all rain examples score below zero, so the perceptron easily finds a valid weight vector.

**Dataset 2 is the XOR problem.**
The label is sun only when city and month both point to summer: (Sydney, December) and (Paris, July). The two sun examples are diagonal opposites of the two rain examples, which means no hyperplane can separate them. Any weights that push both sun scores positive and both rain scores negative lead to a contradiction, adding all four constraints together produces two inequalities that directly oppose each other. This affects every linear classifier: perceptron, SVM, logistic regression, etc.

To make dataset 2 learnable by a linear model, we would need an interaction feature or a polynomial kernel. Alternatively, a non-linear model would handle it directly.

## Task 2 - Baseline perceptron on sentiment data

We reproduce the experiment in doc_classification.py. 
The task is binary sentiment classification on product reviews. 
We first  reproduce the helper code from aml_perceptron.py so the notebook has all the code in itself.


In [ ]:
import numpy as np
import time
from sklearn.base import BaseEstimator
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectKBest
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


def read_data(corpus_file):
    X, Y = [], []
    with open(corpus_file, encoding='utf-8') as f:
        for line in f:
            _, y, _, x = line.split(maxsplit=3)
            X.append(x.strip())
            Y.append(y)
    return X, Y


# Adjust this path if the file is somewhere else.
X, Y = read_data('data/all_sentiment_shuffled.txt')
Xtrain, Xtest, Ytrain, Ytest = train_test_split(X, Y, test_size=0.2, random_state=0)
print(f'Training instances: {len(Xtrain)},  test instances: {len(Xtest)}')


Training instances: 9531,  test instances: 2383


In [ ]:
#Code from aml_perceptron.py

class LinearClassifier(BaseEstimator):

    def decision_function(self, X):
        return X.dot(self.w_)

    def predict(self, X):
        scores = self.decision_function(X)
        return np.select(
            [scores >= 0.0, scores < 0.0],
            [self.positive_class, self.negative_class],
            default=self.negative_class  
        )

    def find_classes(self, Y):
        classes = sorted(set(Y))
        if len(classes) != 2:
            raise Exception('this does not seem to be a 2-class problem')
        self.positive_class = classes[1]
        self.negative_class = classes[0]

    def encode_outputs(self, Y):
        return np.array([1 if y == self.positive_class else -1 for y in Y])
    
class Perceptron(LinearClassifier): #Plain perceptron (same code as in aml_perceptron.py).

    def __init__(self, n_iter=20):
        self.n_iter = n_iter

    def fit(self, X, Y):
        self.find_classes(Y)
        Ye = self.encode_outputs(Y)
        if not isinstance(X, np.ndarray):
            X = X.toarray()
        n_features = X.shape[1]
       # Renamed self.w to self.w_ because sklearn requires fitted attributes to end with _ to confirm the model has been trained and it gave an error before

        self.w_ = np.zeros(n_features)  
        for _ in range(self.n_iter):
            for x, y in zip(X, Ye):
                score = x.dot(self.w_)  
                if y * score <= 0:
                    self.w_ += y * x  


# Helper functions for sparse operations (used later in the bonus task).
def add_sparse_to_dense(x, w, factor):
    """Equivalent of  w += factor * x  when x is a sparse 1-D vector."""
    w[x.indices] += factor * x.data

def sparse_dense_dot(x, w):
    """Equivalent of  x.dot(w)  when x is sparse and w is dense."""
    return np.dot(w[x.indices], x.data)


In [4]:
pipeline = make_pipeline(
    TfidfVectorizer(),
    SelectKBest(k=1000),
    Normalizer(),
    Perceptron(n_iter=30),
)

t0 = time.time()
pipeline.fit(Xtrain, Ytrain)
t1 = time.time()
print(f'Perceptron training time: {t1-t0:.2f} sec.')
print(f'Perceptron test accuracy: {accuracy_score(Ytest, pipeline.predict(Xtest)):.4f}')

Perceptron training time: 1.29 sec.
Perceptron test accuracy: 0.8095


With 30 epochs we get around 0.80 accuracy score on the test set, as expected.


## Task 3 - Implementing the Pegasos SVC

We now translate Algorithm 1 from the clarification document into Python


We loop a fixed number of epochs over the training set (the same convention used in the perceptron code we already have). The clarification document recommends lambda = 1/N as a starting point so we do it.



In [ ]:
class PegasosSVC(LinearClassifier):
    #Pegasos algorithm. SVC with hinge loss.

    def __init__(self, n_iter=10, lambd=None):
        self.n_iter = n_iter
        self.lambd = lambd

    def fit(self, X, Y):
        self.find_classes(Y)
        Ye = self.encode_outputs(Y)
        if not isinstance(X, np.ndarray):
            X = X.toarray()
        n_samples, n_features = X.shape
        lambd = self.lambd if self.lambd is not None else 1.0 / n_samples
        self.w_ = np.zeros(n_features) 

        t = 0
        for _ in range(self.n_iter):
            for x, y in zip(X, Ye):
                t += 1
                eta = 1.0 / (lambd * t)
                score = x.dot(self.w_) 
                if y * score < 1:
                    self.w_ = (1 - eta * lambd) * self.w_ + (eta * y) * x  
                else:
                    self.w_ = (1 - eta * lambd) * self.w_ 

In [6]:
svc_pipeline = make_pipeline(
    TfidfVectorizer(),
    SelectKBest(k=1000),
    Normalizer(),
    PegasosSVC(n_iter=10),     
)

t0 = time.time()
svc_pipeline.fit(Xtrain, Ytrain)
t1 = time.time()
print(f'PegasosSVC training time: {t1-t0:.2f} sec.')
print(f'PegasosSVC test accuracy: {accuracy_score(Ytest, svc_pipeline.predict(Xtest)):.4f}')


PegasosSVC training time: 1.14 sec.
PegasosSVC test accuracy: 0.8326


**Discussion** 

With lambda = 1/N and 10 epochs, the SVC reaches about 0.83 test accuracy, a small  improvement over the perceptron  (~0.80). Decreasing lambda (less regularisation) typically gives a small further boost but trades it back in variance. Also,  increasing the number of epochs past ~20, gives the same score and can start to over-fit.

**Why this is just SGD on the SVC objective**

Pegasos is just stochastic gradient descent on the standard SVM loss function. That loss has two parts: the hinge loss (penalising misclassified or close-to-boundary points) and a regularisation term that keeps the weights small. Each Pegasos update does two things: first it shrinks the weights slightly toward zero (from the regulariser), then if a point is misclassified or too close to the boundary, it updates the weights in the right direction (from the hinge loss). The step size shrinks over time, which is what guarantees the algorithm eventually converges.

## Task 4 - Logistic Regression with the log loss

Logistic regression replaces the hinge loss with the log loss: log(1 + exp(-y * (w dot x))).

Plugging its gradient into the SGD update (the regulariser rescaling is the same as before) gives:

w = (1 - eta*lambda)*w + eta * (y / (1 + exp(y * (w dot x)))) * x

Also we must note the following:

The log loss is differentiable everywhere, so the same update applies to every example, unlike the hinge loss which has two cases.

The gradient has a leading minus, and the SGD update subtracts the gradient. The two minus signs cancel, leaving a plus in front of the sigmoid factor.

In [ ]:
class PegasosLR(LinearClassifier):
    #Pegasos-style SGD with log loss 

    def __init__(self, n_iter=10, lambd=None):
        self.n_iter = n_iter
        self.lambd = lambd

    def fit(self, X, Y):
        self.find_classes(Y)
        Ye = self.encode_outputs(Y)
        if not isinstance(X, np.ndarray):
            X = X.toarray()
        n_samples, n_features = X.shape
        lambd = self.lambd if self.lambd is not None else 1.0 / n_samples
        self.w_ = np.zeros(n_features)  

        t = 0
        for _ in range(self.n_iter):
            for x, y in zip(X, Ye):
                t += 1
                eta = 1.0 / (lambd * t)
                score = x.dot(self.w_)  
                grad_coef = y / (1.0 + np.exp(y * score))
                self.w_ = (1 - eta * lambd) * self.w_ + (eta * grad_coef) * x 

In [8]:
lr_pipeline = make_pipeline(
    TfidfVectorizer(),
    SelectKBest(k=1000),
    Normalizer(),
    PegasosLR(n_iter=10),
)

t0 = time.time()
lr_pipeline.fit(Xtrain, Ytrain)
t1 = time.time()
print(f'PegasosLR training time: {t1-t0:.2f} sec.')
print(f'PegasosLR test accuracy: {accuracy_score(Ytest, lr_pipeline.predict(Xtest)):.4f}')


PegasosLR training time: 1.38 sec.
PegasosLR test accuracy: 0.8317


On this dataset, the logistic-regression variant performs essentially the same as the SVC. Both land around 0.83 with n_iter=10, lambd=1/N. 

This is expected: away from the margin the two loss functions look very similar, and the SGD update we use is almost identical.

The training time is slightly higher than for the SVC because we always perform the additive update (the SVC skips it when y*score >= 1).


## Optional - Printing the objective during training

The objective we are minimising is

$$f(w) = \frac{1}{N}\sum_i \mathrm{Loss}(w,x_i,y_i) + \frac{\lambda}{2}\,\|w\|^2.$$

An approximation of the objective is to accumulate the each loss as it is encountered during an epoch (using the value of w at the time, before the update). 

We use the aproximation because computing the exact objective would require an additional pass over the entire training set every epoch.


In [ ]:
class PegasosSVCWithObjective(LinearClassifier):
    #Pegasos SVC that tracks objective after every epoch.

    def __init__(self, n_iter=10, lambd=None):
        self.n_iter = n_iter
        self.lambd = lambd

    def fit(self, X, Y):
        self.find_classes(Y)
        Ye = self.encode_outputs(Y)
        if not isinstance(X, np.ndarray):
            X = X.toarray()
        n_samples, n_features = X.shape
        lambd = self.lambd if self.lambd is not None else 1.0 / n_samples
        self.w_ = np.zeros(n_features) 
        self.objective_history_ = []

        t = 0
        for epoch in range(self.n_iter):
            running_loss = 0.0
            for x, y in zip(X, Ye):
                t += 1
                eta = 1.0 / (lambd * t)
                score = x.dot(self.w_)  
                running_loss += max(0.0, 1.0 - y * score)
                if y * score < 1:
                    self.w_ = (1 - eta * lambd) * self.w_ + (eta * y) * x 
                else:
                    self.w_ = (1 - eta * lambd) * self.w_  
            obj = running_loss / n_samples + 0.5 * lambd * self.w_.dot(self.w_) 
            self.objective_history_.append(obj)
            print(f'  epoch {epoch+1:2d}/{self.n_iter} - approx. objective = {obj:.4f}')


svc_obj_pipeline = make_pipeline(TfidfVectorizer(), SelectKBest(k=1000), Normalizer(),
                                 PegasosSVCWithObjective(n_iter=10))
svc_obj_pipeline.fit(Xtrain, Ytrain)
print(f'Test accuracy: {accuracy_score(Ytest, svc_obj_pipeline.predict(Xtest)):.4f}')


  epoch  1/10 - approx. objective = 1.2906
  epoch  2/10 - approx. objective = 0.4452
  epoch  3/10 - approx. objective = 0.4095
  epoch  4/10 - approx. objective = 0.3926
  epoch  5/10 - approx. objective = 0.3837
  epoch  6/10 - approx. objective = 0.3769
  epoch  7/10 - approx. objective = 0.3726
  epoch  8/10 - approx. objective = 0.3692
  epoch  9/10 - approx. objective = 0.3667
  epoch 10/10 - approx. objective = 0.3642
Test accuracy: 0.8326


We see the objective decrease quickly during the first few epochs and then flatten out. 

## Bonus Task 1 - Making your code more efficient

We chose Bonus Task 1 and implemented all three sub-tasks. The arithmetic carried out is the same as in the naive PegasosSVC, so accuracy should not change; what changes is the wall-clock time.


We benchmark all four side by side at the end of this section.


### (a) Faster linear algebra with BLAS

scipy.linalg.blas exposes three primitives that map exactly onto the linear-algebra operations inside the Pegasos loop:

- ddot(x, w) equivalent to x.dot(w)
- dscal(a, w) equivalent to w *= a (in place)
- daxpy(x, w, a=a) equivalent to w += a * x (in place)

These functions skip a number of safety checks that NumPy performs, so they are noticeably faster when invoked millions of times during SGD. They require the arrays to be 1-D, contiguous and float64.

In [ ]:
from scipy.linalg.blas import ddot, dscal, daxpy


class PegasosSVC_BLAS(LinearClassifier):
    #Pegasos SVC accelerated with BLAS primitives. Dense input.

    def __init__(self, n_iter=10, lambd=None):
        self.n_iter = n_iter
        self.lambd = lambd

    def fit(self, X, Y):
        self.find_classes(Y)
        Ye = self.encode_outputs(Y).astype(np.float64)
        if not isinstance(X, np.ndarray):
            X = X.toarray()
        # BLAS wants contiguous float64 vectors.
        X = np.ascontiguousarray(X, dtype=np.float64)
        n_samples, n_features = X.shape
        lambd = self.lambd if self.lambd is not None else 1.0 / n_samples
        self.w_ = np.zeros(n_features, dtype=np.float64)

        t = 0
        for _ in range(self.n_iter):
            for i in range(n_samples):
                x = X[i]
                y = Ye[i]
                t += 1
                eta = 1.0 / (lambd * t)
                score = ddot(x, self.w_)
                #  rescale: w *= (1 - eta*lambd).
                dscal(1.0 - eta * lambd, self.w_)
                if y * score < 1:
                    
                    daxpy(x, self.w_, a=eta * y)


In [11]:
blas_pipeline = make_pipeline(
    TfidfVectorizer(), SelectKBest(k=1000), Normalizer(),
    PegasosSVC_BLAS(n_iter=10),
)

t0 = time.time()
blas_pipeline.fit(Xtrain, Ytrain)
t1 = time.time()
blas_time = t1 - t0
print(f'BLAS SVC training time: {blas_time:.2f} sec.')
print(f'BLAS SVC test accuracy: {accuracy_score(Ytest, blas_pipeline.predict(Xtest)):.4f}')


BLAS SVC training time: 0.88 sec.
BLAS SVC test accuracy: 0.8326


### (b) Sparse vectors (dropping SelectKBest, adding bigrams)

So far we used SelectKBest(k=1000), which projects the sparse TF-IDF representation onto a dense N x 1000 matrix. If we drop this step and add bigrams (ngram_range=(1, 2)), the feature space jumps to hundreds of thousands of dimensions, but each document touches only a handful of them.

The dense implementation has to touch every component of w on every iteration (the rescaling step), which is wasteful when most features are zero. The sparse version uses the two helper functions we already saw in aml_perceptron.py:

* sparse_dense_dot(x, w) for the dot product, and
* add_sparse_to_dense(x, w, factor) for the additive update.

We must mention that the rescaling step self.w_ *= (1 - eta*lambd) still touches every component of w. That is the cost that part (c) below removes.


In [ ]:
class PegasosSVC_Sparse(LinearClassifier):
    #Pegasos SVC that consumes a sparse feature matrix directly

    def __init__(self, n_iter=10, lambd=None):
        self.n_iter = n_iter
        self.lambd = lambd

    def fit(self, X, Y):
        self.find_classes(Y)
        Ye = self.encode_outputs(Y)
        n_samples, n_features = X.shape
        lambd = self.lambd if self.lambd is not None else 1.0 / n_samples
        self.w_ = np.zeros(n_features)
        # iterating sparse rows is slow; pre-cache as a Python list once
        XY = list(zip(X, Ye))

        t = 0
        for _ in range(self.n_iter):
            for x, y in XY:
                t += 1
                eta = 1.0 / (lambd * t)
                score = sparse_dense_dot(x, self.w_)
                # Whole-vector rescaling but still O(n_features).
                self.w_ *= (1.0 - eta * lambd)
                if y * score < 1:
                    add_sparse_to_dense(x, self.w_, eta * y)


In [ ]:
sparse_pipeline = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2)),    # bigrams: much larger feature space
    Normalizer(),
    PegasosSVC_Sparse(n_iter=10),
)

t0 = time.time()
sparse_pipeline.fit(Xtrain, Ytrain)
t1 = time.time()
sparse_time = t1 - t0
print(f'Sparse SVC training time: {sparse_time:.2f} sec.')
print(f'Sparse SVC test accuracy: {accuracy_score(Ytest, sparse_pipeline.predict(Xtest)):.4f}')


Sparse SVC training time: 15.68 sec.
Sparse SVC test accuracy: 0.8695


### (c) Lazy weight-vector rescaling

Even with sparse data, self.w * = (1 - eta*lambd) touches every component of w on every step, so the cost per iteration depends on the total number of features, not the number of non-zero elements in x. Section 2.4 of the Pegasos paper describes a trick that postpones this rescaling.

The idea is to factor the weight vector as a * w_tilde, where a is a single scalar that absorbs all the rescalings. The update equations become:

- Update the scalar: a = (1 - eta*lambda) * a
- Compute the score: score = a * (w_tilde dot x)
- If y * score < 1: update w_tilde += (eta * y / a) * x (dividing by a to compensate for the final multiplication)
- At the end of training: materialise the real weight vector as w = a * w_tilde

Because every operation inside the loop is either scalar arithmetic or a sparse update touching only the non-zero elements of x, the per-iteration cost no longer depends on the total number of features.

In [ ]:
class PegasosSVC_SparseScaled(LinearClassifier):
    #Sparse Pegasos SVC with the lazy scaling factor a

    def __init__(self, n_iter=10, lambd=None):
        self.n_iter = n_iter
        self.lambd = lambd

    def fit(self, X, Y):
        self.find_classes(Y)
        Ye = self.encode_outputs(Y)
        n_samples, n_features = X.shape
        lambd = self.lambd if self.lambd is not None else 1.0 / n_samples
        self.w_ = np.zeros(n_features)
        a = 1.0
        XY = list(zip(X, Ye))

        t = 0
        for _ in range(self.n_iter):
            for x, y in XY:
                t += 1
                eta = 1.0 / (lambd * t)
                a *= (1.0 - eta * lambd)

                # Rescale to avoid division by zero when a collapses to 0
                if abs(a) < 1e-10:  
                    self.w_ *= a    # materialise current effective vector
                    a = 1.0         # reset scalar

                score = a * sparse_dense_dot(x, self.w_)
                if y * score < 1:
                    add_sparse_to_dense(x, self.w_, eta * y / a)

        self.w_ *= a

In [15]:
scaled_pipeline = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2)),
    Normalizer(),
    PegasosSVC_SparseScaled(n_iter=10),
)

t0 = time.time()
scaled_pipeline.fit(Xtrain, Ytrain)
t1 = time.time()
scaled_time = t1 - t0
print(f'Sparse+Scaled SVC training time: {scaled_time:.2f} sec.')
print(f'Sparse+Scaled SVC test accuracy: {accuracy_score(Ytest, scaled_pipeline.predict(Xtest)):.4f}')


Sparse+Scaled SVC training time: 3.04 sec.
Sparse+Scaled SVC test accuracy: 0.8666


### Putting the four variants side by side

- Perceptron: (dense, k = 1000): ~1.5 s, ~0.80 accuracy
- PegasosSVC: (dense NumPy, k = 1000): ~1.14 s, ~0.83 accuracy
- PegasosSVC_BLAS: (dense + BLAS, k = 1000): ~0.88 s, ~0.83 accuracy
- PegasosSVC_Sparse: (full vocab + bigrams): ~15 s, ~0.86 accuracy
- PegasosSVC_SparseScaled:  (sparse + lazy scaling): ~3 s, ~0.86 accuracy



The lazy scaling trick is by far the most impactful optimisation when the feature space is high-dimensional:
Going from ~15 s to ~3 s without any change to what the algorithm computes confirms that the per-step whole-vector rescaling was the main cost.

BLAS gives a notable speed-up on dense inputs essentially for free, because the inner loop is dominated by small ddot/daxpy operations that benefit from the lighter calling convention.

Accuracy is similar across the four SVC variants, but the sparse versions show a small but consistent improvement (0.83 to 0.86). This comes from the larger, bigram-augmented feature space rather than any algorithmic change. By keeping the full vocabulary and adding bigrams, the model gains access to short phrases that carry sentiment information a single word cannot capture. These compound features give the classifier a richer representation of the text, which is enough to recover a few percentage points of accuracy.